# Baseball Lab 3: Assessing Impressive Statistics and Comparing Players

In today's lab exercises you'll practice analyzing and visualizing data by assessing impressive statistics using percentiles, five-number summaries, and box plots. You'll also practice comparing players from different time periods and writing functions to calculate impressive statistics. 

Please complete this notebook by filling in the cells provided. For all problems that you must write explanations and sentences for, please provide your answer in the designated space. 


#### Deadline

This assignment is due **Sunday February 8th at 11pm**. You can turn in the assignment up to 24 hours late for 90% credit (after that, the homework will only be accepted with a Dean's Extension). Directly sharing answers is not okay, but discussing problems with the course staff or with other students is encouraged. Refer to the policies page to learn more about how to learn cooperatively. You should start early so that you have time to get help if you're stuck. If you have questions, please post them to [Ed Discussion](https://edstem.org/us/courses/89102/discussion) (and answer others' questions too). 


## Getting started - downloading the data

In order to complete this lab, it is necessary to download a few files. Please run the code below **only once** to download data needed to complete the lab. To run the code, click in the cell below and press the play button (or press shift-enter). 


In [22]:
# Please run this code once to download the files you will need to complete the homework 

import YData_baseball

YData_baseball.download_data("Lahman_2024u/Batting.csv")
YData_baseball.download_data("Lahman_2024u/People.csv")
YData_baseball.download_data("Lahman_2024u/Pitching.csv")

The file `Batting.csv` already exists.
If you would like to download a new copy of the file, please rename the existing copy of the file.
The file `People.csv` already exists.
If you would like to download a new copy of the file, please rename the existing copy of the file.
The file `Pitching.csv` already exists.
If you would like to download a new copy of the file, please rename the existing copy of the file.


# Part 0: Quote and reaction to Astroball chapter 3 (5 points)

Please find an interesting quote from chapter 3 of Astroball (preface or prologue) and then write a ~one paragraph reaction to the quote below.

*Quote:*  ...

Reaction: ... 

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Part 1: Assessing which statistics are "good"

As discussed in class, using percentiles is one way to assess what value is impressive for a particular statistic. In the first set of exercises, let's try to determine what are "impressive" statistic value for a range of popular statistics that are used to evaluate baseball batters. At first we will focus on what an impressive value is for home runs, and later we will try to find impressive statistics values for other popular baseball statistics. 

Below we load the Lahman Batting data which is the data we will use for these exercises. 


In [24]:
batting = pd.read_csv("Batting.csv")

batting.head()



,playerID,yearID,stint,teamID,lgID,G,G_batting,AB,R,H,...,SB,CS,BB,SO,IBB,HBP,SH,SF,GIDP,G_old
0,aardsda01,2004,1,SFN,NL,11,NaN,0,0,0,...,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
1,aardsda01,2006,1,CHN,NL,45,NaN,2,0,0,...,0.0,0.0,0,0.0,0.0,0.0,1.0,0.0,0.0,NaN
2,aardsda01,2007,1,CHA,AL,25,NaN,0,0,0,...,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
3,aardsda01,2008,1,BOS,AL,47,NaN,1,0,0,...,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,NaN
4,aardsda01,2009,1,SEA,AL,73,NaN,0,0,0,...,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,NaN


## 1.1 Augmenting our data

When analyzing data, it is often useful to clean and wrangle the data before one gets started analyzing the data. For these exercises, let's start by augmenting the Lahman batting data with additional columns that are derived from existing columns. We will do this at the beginning of the analyses so that we can use a consistent data set throughout the analyses. 

**Question 1.1 (5 points)**  Please create a new DataFrame called `batting_enhanced` that has all the original columns in the `batting` DataFrame, adds has the following additional columns: 

1. `PA`: The number of plate appearances (PA) for each player. A plate appearance is defined as PA = AB + BB + HBP + SH + SF, where AB is at-bats, BB is walks, HBP is hit by pitch, SH is sacrifice hits, and SF is sacrifice flies. When adding sacrifices flies (SF), please fill in missing values as 0 using the `fillna(0)` method. 

2. `AVG`: The batting average (AVG) for each player, which is defined as AVG = H / AB, where H is the number of hits and AB is the number of at-bats. 

3. `SLG`: The slugging percentage (SLG) for each player, which is defined as SLG = (1B + 2* 2B + 3* 3B + 4 * HR) / AB, where H is the number of hits, 2B is the number of doubles, 3B is the number of triples, HR is the number of home runs, and AB is the number of at-bats. 

4. `OBP`: The on-base percentage (OBP) for each player, which is defined as OBP = (H + BB + HBP) / (AB + BB + HBP + SF), where H is the number of hits, BB is the number of walks, HBP is the number of times hit by pitch, AB is the number of at-bats, and SF is the number of sacrifice flies. 

5. `OPS`: The on-base plus slugging percentage (OPS) for each player, which is defined as OPS = OBP + SLG, where OBP is the on-base percentage and SLG is the slugging percentage. 

To make sure your derived statistics are calculated correctly, there is test code below that compares the derived statistics for Shohei Ohtani in the `batting_enhanced` DataFrame to the expected values. Since it is important that you get the derived statistics correct, we have written the test code for you and **do not proceed** to the following exercises if any of the test code below returns false. In general when doing your own analyses you would have to write your own test code, which you could check by looking at known validated sources (e.g., https://www.baseball-reference.com/players/o/ohtansh01.shtml) 


In [16]:

batting_enhanced = batting.copy()

# Add the additional columns to the batting_enhanced DataFrame







# Test that the additional columns were added correctly

shohei_ohtani_2021 = round(batting_enhanced.query("playerID == 'ohtansh01' and yearID == 2021")[["PA", "AVG", "SLG", "OBP", "OPS"]].squeeze(), 3)

shohei_ohtani_2021 == pd.Series({"PA": 639, "AVG": 0.257, "SLG": 0.592, "OBP": 0.372, "OPS": .965})


## 1.2 Percentiles

Recall that the $p^{th}$ percentile is the value of a quantitative variable which is greater than *p* percent of the data. We can get the value the $p^{th}$ percentile from a pandas Series using `.quantile()` method. For example, we can get the 80th percentile value using: `my_df["col"].quantile(.8)`. Alternatively, we could use `np.percentile(my_df["col"], 90)` to get the same result. 

**Question 1.2 (5 points)**: What is the 90th percentile value for home runs hit? Does this value seem reasonable to assess what a good player is today? 

For this and all questions below, please write code to calculate the answer and print out the relevant values to "show your work". Then in the answer cell below, please write down the answers to the questions asked (e.g., for this question, answer whether this value seems reasonable to assess whether a player is impressive).


In [25]:
# Calculate the 90th percentile of home runs hit using all the data in the batting DataFrame






**Answer**: 





## 1.3 Percentiles using only players since the year 2000

One reason that using percentiles on all the Lahman batting data might not be a good way to find what is an "impressive" statistic value for the number of home runs is due to the fact that perhaps the game of baseball has changed over time. Let's explore this by finding the 90th percentile value for only players who played since the year 2000.

**Question 1.3 (5 points):**  What is the 90th percentile value for home runs since 2000? Is it much different than compared to using all years going back to 1871? Do you think this value accurately represents how many home runs an impressive player hits today? 

Hint: using Boolean indexing (or the `.query()` method) to filter the data will be useful.


**Answer** 





## 1.4 Percentiles using only players that have 502 or more plate appearances 

Another possible reason this method might not be finding an "impressive" statistic value for the number of home runs is due to the fact that many players only play a few games in a season and thus do not have that many opportunities to hit home runs. Indeed, to qualify as a leader in a batting statistic, you need to have at least 502 plate appearances, so let's try our analysis using only players who have at least 502 plate appearances. 


**Question 1.4 (5 points):**  Please filter the batting DataFrame to only include players that have at least 502 at-bats in a season and save this result to the name `players_502PA`. Then calculate the 90th percentile value for home runs using this filtered DataFrame. Is it much different compared to using all players in the Lahman Batting file? In the answer section, please write down whether you think this value accurately represents how many home runs an impressive player hits today. 


**Answer** 




## 1.5 Five number summary

Recall that the five number summary consists of: 
1. The minimum value
2. The 25th percentile value (Q1)
3. The 50th percentile value (the median)
4. The 75th percentile value (Q3) 
5. The maximum value

**Question 1.5 (5 points):** What is the 5 number summary when using data from all players (using the `batting_enhanced` DataFrame) and when using data from only players with 502 or more at-bats (using the `batting_502PA` DataFrame)? Be sure to print out the Python code that calculates these results below and in the answer section write down these results and describe which numbers are different and which are the same. 


**Answer** 

The five-number summary for all players is: 

The five-number summary for players with at least 502 at-bats is: 

...

## 1.6 Boxplots 

Boxplots are a visualization of the five-number summary. We can create boxplots using the Matplotlib library's `plt.boxplot()` function and passing the function a sequence of numbers (e.g., an list, ndarray, or Series of numbers). 

**Question 1.6 (5 points):** Create a boxplot for home runs using only data for players that have 502 or more at-bats (and be sure to label your axes). Based on this boxplot, would the 34 home runs George Springer hit in 2017 be considered an outlier? 


In [26]:
import matplotlib.pyplot as plt









**Answer** 



## 1.7 Histograms

Histograms are a convenient way to view the full shape of a distribution. We can create a histogram using Matplotlib's `plt.hist()` function, where we pass the function a sequence of numeric data we want to plot as a histogram. 

**Question 1.6 (5 points):** Create a histogram of the home runs hit by players who have 502 or more at-bats in a season. Also, add a red vertical line at the 90th percentile value for home runs hit by players with 502 or more at-bats using the `plt.axvline()` function. As always, sure to label your axes. 

Bonus: see if you can figure out how to create bins in the histogram such that the first bin has the number of players who hit 0 home runs, the second bin has the number of players who hit 1 home run, etc. and the y-axis displays the counts of the number of home runs.


## 1.8 Writing a function to calculate "impressive" values for any baseball statistic

Let's now write a function that can calculate what an impressive statistic is for any statistic (or at least, any statistics in the `batting_enhanced` DataFrame). This will allow us to easily find what is an "impressive" value for any statistic. 

**Question 1.8 (12 points):** Please write a function called `get_percentile_statistics()` that takes the following arguments: 

1. `statistic`: A string, or list of strings, that specify the batting statistics that we should get a percentiles of. This argument should have the same name(s) as columns in the a Lahman-like batting DataFrame (e.g.,  "HR", ["2B", "SB"]). 

2. `percentiles`: A number, or list of numbers, between 0 and 100 specifying the percentile(s) to calculate. For example, if you want to calculate the 90th percentile, you would pass in `percentiles = 90`. If you want to calculate the 25th, 50th, and 75th percentiles, you would pass in `percentiles = [25, 50, 75]`

3. `min_PA`: A number specifying the minimum number of plate appearances that a player must have to be included in the calculation of the percentiles. For example, if you want to only include players with at least 502 plate appearances, you would pass in `min_PA = 502`

4. `year_range`: A tuple specifying the starting year and ending year of the range of years to include in the calculation. For example, if you want to only include players from 2000 to 2024, you would pass in `year_range = (2000, 2024)`

Your function should operate on the `batting_enhanced` data, and return the percentile values for the specified statistics either a floating point number (if a single percentile value was given) or as a pandas Series/DataFrame if a list/ndarray of multiple percentile values and/or batting statistics is given.

Once your function is written, please run the test code below to check that your function works correctly. If the test code returns `True`, then your function is working correctly. If it returns `False`, then you should debug your function until it returns `True`. 


In [27]:
def get_percentile_statistics(statistic, percentiles, min_PA, year_range):
    """
    Calculate the specified percentiles for a given statistic in a Lahman-like batting DataFrame.
    
    Parameters:
    - statistics: str or list of str, the name of the statistic(s) column(s) to analyze (e.g., 'HR', 'SB').
    - percentile: int or list of int, the percentile(s) to calculate (e.g., 90 or [25, 50, 75]).
    - min_PA: int, the minimum number of plate appearances required to include a player.
    - year_range: tuple of int, the range of years to filter the data (start_year, end_year).
    
    Returns:
    - dict: A dictionary with percentiles as keys and their corresponding values.
    """
    
   # Fill in the function body here...













# Test the function 
print(np.sum(get_percentile_statistics(["HR", "SB"], 50, 502, (2000, 2024)) == np.array([20, 7])) == 2)


False


## 1.9 Assessing impressive statistics

Now let's use the `get_percentile_statistics()` function to assess what impressive statistic values are for several population batting statistics.

**Question 1.9 (5 points):**  Please use the `get_percentile_statistics()` function to calculate the following statistics for players with at least 502 plate appearances from 2000 to 2024: `HR`, `AVG`, `SLG`, `OBP`, `OPS`, and `SB`. 

For each statistic, calculate the following percentiles: 

- The 10th percentile (i.e., the value that is greater than 10% of the players) which would indicate a poor performance
- The the median (50th percentile), which would indicate average performance)
- The 90th percentile (i.e., the value that is greater than 90% of the players) which would indciate very impressive performance  

Using the `get_percentile_statistics() in cell below, please write **one line of code** that calculates these statistics and prints the results.


## 1.10 How impressive is Ohtani?

Finally, let's compare these "impressive" batting statistic values to Ohtani's batting statistics to see how impressive he is.

**Question 1.10 (5 points):**  Please compare the results you calculated above to Shohei Ohtani in the 2024 season. You can find Ohtani's statistics in the `batting_enhanced` DataFrame by filtering using his playerID which is `ohtansh01`. In the answer section, please write down which of Ohtani's statistics are impressive.


**Answer**








# Part 2: Comparing players from different time periods

In class we compared several baseball players to see who was most impressive. Let's continue this type of analysis by comparing Babe Ruth and Shohei Ohtani. 


## 2.1 Box plot comparison 

To begin, let's create a box plot comparing the number of home runs hit by Shohei Ohtani and Babe Ruth. 

**Question 2.1 (5 points):**  Please create a box plot comparing the number of home runs hit by Shohei Ohtani and Babe Ruth. You should use the `players_enhanced` DataFrame and filter it to get data for boxplots. In particular, please complete the following steps to create this boxplot:

1. Create a DataFrame called `ohtani_data` that contains only the data for Shohei Ohtani, and set the index of this DataFrame to be the `yearID` column. 
 
2. Create a DataFrame called `ruth_data` that contains only the data for Babe Ruth, and set the index of this DataFrame to be the `yearID` column. 

3. Create a box plot comparing the number of home runs hit by Ohtani and Ruth using the `plt.boxplot()` function. Be sure to label your axes appropriately.

In the answer section, write down who seems more impressive based on the box plot and briefly mention any limitations to the visualization. 


**Answer** 








# 2.2 Visualizing individual season home runs

As we also discussed in class, box plots are not always the best way to visualize data when there are only a few data values. Instead, we can use a beeswarm plot to visualize the individual home runs hit by Ohtani and Ruth in each season. 

**Question 2.2 (5 points):**  Please use `seaborn` to create a beeswarm plot comparing the number of home runs hit by Ohtani and Ruth in each season. You can use the `sns.catplot()` function to create the beeswarm plot. Be sure to label your axes appropriately. 


In the answer section, write down who seems more impressive based on the box plot and again discuss some of the limitations of the plot. 


**Answer** 








# 2.3 Compare HR over years

As we discussed in class, comparing statistics across different time periods could be misleading because the game of baseball could have changed over time. Let's examine if the number of home runs typically hit has changed across time by plotting the number of home runs hit by the most prolific home run hitter in each year. 

**Question 2.3 (5 points):**  Please create a line plot showing the maximum number of home runs hit by any player in each year from 1871 to 2024. Your plot should include both a line connecting points for each year, and well as individual dots showing the maximum home runs hit in each year. Also, as always, be sure to label your axes appropriately. 

In the answer section please answer the following questions: 

a. Is there a trend and/or a specific time period where the number of home runs hit changed significantly? 

b. Are there any noticeable outlier seasons where the maximum number of home runs hit by any player was significantly different than the surrounding seasons? If so, please explain what might account for some of these outlier seasons. 


**Answers** 

a. 


b.





## 2.4 Looking at the maximum number of home runs hit when Ruth and Ohtani were playing

To get a better sense of how the how many home runs were typically hit **during the periods of time when Ruth and Ohtani were playing**, please recreate the line plot in question 2.3 but add following vertical lines to the plot: 

1. A vertical line  (using the `plt.axvline()` function) at the first year Ruth played and a vertical line at the last year Ruth played. These lines should be red dashed lines with a label of "Ruth" for Ruth's first year. Also make the lines dashed, and make the alpha transparency value 0.5 so that the lines are not too dark. 

2. A vertical line at the first year Ohtani played and a vertical line at the last year Ohtani played. These lines should be green dashed lines with a label of "Ohtani for Ohtani's first year. Also make the lines dashed, and make the alpha transparency value 0.5 so that the lines are not too dark.

3. Call the `plt.legend()` function to show the legend indicating which vertical lines correspond to Ruth and which to Ohtani. 

From looking at this plot, does it appear the maximum number of home runs hit by any player has changed during the time periods when Ruth and Ohtani were playing? Also, are there any features of the plot that could be misleading and/or give some insights into any of the plots you created above? 
    

**Answer** 











## 2.5 Z-scores of maximum home runs hit

As we also discussed in class, to assess how impressive a statistic value is, it is useful to compare the value to the distribution of values for that statistic produced by other players who were playing at the same time, and one way to do this is to calculate the z-scores. As we discussed, a z-score is defined as: 

$z_i = \frac{x_i - \bar{x}}{s}$

where: 
- $x_i$ is the original statistic value (e.g., home runs hit)
- $\bar{x}$ is the mean value for that statistic (e.g., mean number of home runs hit by all players in a given year).
- and $s$ is the standard deviation of the distribution of values for that statistic. 


**Question 2.5 (5 points):**  Please calculate the z-scores of the maximum number of home runs hit by any player for each year (i.e., there will be a separate z-scores for each year, so you will end up with a Series of z-scores). To do this, please use the following steps: 

Please convert the maximum number of home runs hit by any player in each year to z-scores using the following steps: 

1. Calculate the mean number of home runs hit each year using the `players_502PA` DataFrame (so that only players with at least 502 plate appearances are included in the calculation) as a pandas Series and store the results in a variable called `mean_hr`. 

2. Calculate the standard deviation of the number of home runs hit each year again using the `players_502PA` DataFrame as a pandas Series and store the results in a variable called `sd_hr`.

3. Calculate the maximum number of home runs hit each year using the `players_502PA` DataFrame as a pandas Series and store the results in a variable called `max_hr`. 

4. Convert the maximum number of home runs hit each year (in the `max_hr` Series) to z-scores using the formula above and save the results in a variable called `zscore_max_HR`. 

5. Create a scatter plot showing these z-scores are a function of the year.

6. Add vertical lines to the plot at the first and last year Ruth played, as well as the first and last year Ohtani played. Use red dashed lines for Ruth and green dashed lines for Ohtani, and label the lines appropriately. 

In the answer section report if there are any noticeable particular high z-scores during the seasons Ruth and/or Ohtani played, and speculation of which player(s) might have produced these high z-scores. 


**Answer**






## 2.6 Comparing z-scores of Ohtani and Ruth

Let's now compare how good Ohtani and Ruth were in terms of z-scores (i.e., who was most impressive relative to other players who played at the same time). 


**Question 2.6 (5 points):**  Please compare the z-scores of the number of home runs hit by Ohtani and Ruth in each year they played by doing the following: 

1. Use the `ohtani_data` and the `mean_hr` and `sd_hr` data, to calculate the z-scores of the number of home runs hit by Ohtani each year he played. Remove all missing values from this data using using the `dropna()` method and save this result to a variable called `zscores_ohtani`. 

2.  Use the `ruth_data` and the `mean_hr` and `sd_hr` data, to calculate the z-scores of the number of home runs hit by Ruth each year he played. Remove all missing values from this data using using the `dropna()` method and save this result to a variable called `zscores_ruth`. 

3. Create side-by-side boxplots of the z-scores of Ohtani and Ruth. 

In the answer section report:

Whose z-scores are most impressive, and based on this, who do you think is the more impressive player? 


**Answer**









## 2.7 Finding the year's where Ruth and Ohtani had their best performance relative to their peers

Let's find the year's where Ruth and Ohtani had their best performance relative to their peers. 

**Question 2.7 (5 points):**  Please find the year where Ruth hit the most home runs relative to his peers (i.e., the year with the highest z-score) and save the result to `ruth_best_year`. Likewise, find the year where Ohtani hit the most home runs relative to his peers (i.e., the year with the highest z-score). Print out these years to show your work.

Hint: One way to do this is the use the `.reset_index()` method to convert the yearID back to a column, and then use the `.sort_values()` method to sort the DataFrame by the z-scores. Then one can get the first row of the DataFrame using `.iloc[0]` and then get the yearID from this row.


## 2.8 Visualizing the number of home runs hit in Ohtani and Ruth's best years

Finally, let's visualize the number of home runs hit in Ruth and Ohtani's best year using beeswarm plots. 

**Question 2.8 (5 points):**  Please create beeswarm plots showing the number of home runs hit in Ohtani best year and a beeswarm plot showing the number of home runs hig in Ruth's best year. To do this, please do the following: 

1. Create a DataFrame called `data_ohtani_best_year` that contains only the data from Ohtani's best year and only from players that had 502 plate appearances (i.e., use the `players_502PA`). Then create a beeswarm plot of home runs hit for all players using the `sns.catplot()` function.

2. Repeat this for Ruth's best year (but save the DataFrame to `data_ruth_best_year`).

In the answer section, please write down any impressions you have of either of these plots. 


**Answer**








# Part 3: Try some additional analyses on your own (10 points)

Please try to run some additional analyses on your own that give you additional insight into impressive players. A couple of possibilities are:

1. Data on pitchers is loaded below. What are some impressive pitching statistics (stike outs, ERA, etc), and who are some outstanding pitchers?
2. Compare Babe Ruth to another hitter. Is there another hitter who is more impressive than Ruth?
3. Come up with another analysis of your own (e.g., you could look at the performance of particular players at particular ages, etc.). 


In [28]:

pitchers = pd.read_csv("Pitching.csv")
pitchers.head()





,playerID,yearID,stint,teamID,lgID,W,L,G,GS,CG,...,IBB,WP,HBP,BK,BFP,GF,R,SH,SF,GIDP
0,aardsda01,2004,1,SFN,NL,1,0,11,0,0,...,0.0,0.0,2.0,0.0,61.0,5.0,8,0.0,1.0,1.0
1,aardsda01,2006,1,CHN,NL,3,0,45,0,0,...,0.0,1.0,1.0,0.0,225.0,9.0,25,1.0,3.0,2.0
2,aardsda01,2007,1,CHA,AL,2,1,25,0,0,...,3.0,2.0,1.0,0.0,151.0,7.0,24,2.0,1.0,1.0
3,aardsda01,2008,1,BOS,AL,4,2,47,0,0,...,2.0,3.0,5.0,0.0,228.0,7.0,32,3.0,2.0,4.0
4,aardsda01,2009,1,SEA,AL,3,6,73,0,0,...,3.0,2.0,0.0,0.0,296.0,53.0,23,2.0,1.0,2.0


# 3. Reflection (3 points)

Please fill out the lab 3 reflection on Canvas to let us know how this lab, and the class overall, is going for you. 



# 4.  Submitting your work

Once you're finished filling in and running all cells, you should submit your assignment as a pdf on Gradescope. You can access Gradescope through Canvas on the left-side of the class home page. The problems in each lab assignment are numbered. When submitting on Gradescope, please **make sure to select the correct pages of your pdf that correspond to each problem**. Failure to mark pages correctly **will result in points being deducted** from your score.

To convert this Jupyter notebook document to a pdf please run the code in the cell below. This should produce a document called `lab_02.pdf` which should appear in the files tab on the left (you might need to refresh the files tab to see this file). You can then right click on this file (command click on a mac), to download this pdf document which you can upload to Gradescope. 

Please be sure to check that all the code and output are visible before submitting your pdf to Gradescope as **points will be deducted for missing code and output that is not visible** since we will not be able to grade this.

In [29]:
%%capture

!quarto render lab_03.ipynb --cache-refresh --to pdf 

#### Alternative submission instructions

If converting your Jupyter notebook to a pdf using the command in the cell above does not work, an alternative way to convert your Jupyter notebook is:

1.  Go to "File" at the top-left of your Jupyter Notebook
2.  Under "Download as" (or "Save and Export Notebook As...") and select "HTML (.html)"
3.  After the .html has downloaded, open it and then select "File" and "Print" (note you will not actually be printing)
4.  From the print window, select the option to save as a .pdf
